# Known-item retrieval evaluation

Quantitative metrics for the Glance fashion retriever (**Recall@K, MRR, median rank**).

## Why known-item retrieval?
There is **no human-labeled `query → relevant image` set** in this project. The honest,
fully-automated way to get real numbers is *known-item* (a.k.a. self-)retrieval:

> For each indexed image, write a natural-language query describing it, then measure **where that
> exact image lands in the ranking.** Relevance = the source image itself (one relevant item/query).

This scores the **embedding + late-fusion + rerank** pipeline end-to-end.

**Leakage control.** The *indexed* caption is the full `attrs_to_text` sentence (garments + shoes +
accessories + action + style + environment). The *eval query* is a deliberately **shortened**
description built from the structured metadata — primary garment + colour (`q_short`), optionally
plus environment (`q_medium`). So the query is **not** the stored string verbatim; it stands in for a
realistic short user query rather than a trivial exact match.

**What this does and does not measure.** It measures *"can a short natural description of an image
retrieve that image?"* — i.e. representation/fusion quality. It is **not** a measure of absolute
production accuracy against real user intent (that needs human judgments). State this in the write-up.

Retrieval is driven through the package's `FusionRetriever` (single source of truth) — nothing is
re-implemented here except the query synthesis and the metrics.

In [1]:
import os, sys, glob, json
from statistics import median

import numpy as np
import pandas as pd

# make the package importable when running from Notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "Notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from fashion_retrieval.configs.config import CONFIG as cfg
from fashion_retrieval.retriever.retrieve import FusionRetriever
from fashion_retrieval.retriever.fusion import fused_candidates
from fashion_retrieval.retriever.rerank import rerank

r = FusionRetriever()                 # loads SigLIP + text.faiss + image.faiss + records.jsonl
records = r.records
N = len(records)
print(f"{N} indexed records; image index ntotal = {r.image_index.ntotal}")
print(f"defaults: top_candidates={cfg.top_candidates}, fusion_alpha={cfg.fusion_alpha}, "
      f"rerank_enabled={cfg.rerank_enabled} (weight={cfg.rerank_weight})")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


461 indexed records; image index ntotal = 461
defaults: top_candidates=100, fusion_alpha=0.5, rerank_enabled=True (weight=0.15)


## 1. Synthesize short queries from structured metadata

We read the raw Qwen3-VL metadata (`data/qwen3_vl_metadata/<stem>.json`), which keeps colour and
garment type as separate fields, and build a short query per image:

- `q_short`  = primary garment `"{colour} {type}"`, e.g. *"beige sleeveless dress"*
- `q_medium` = `q_short` + `" in {environment}"`

Primary garment = first non-empty of **dress -> top -> bottom -> outerwear**. Records with no garment
at all can't form a query and are skipped (counted below).

In [2]:
META_DIR = os.path.join(ROOT, "data", "qwen3_vl_metadata")
meta_by_id = {}
for fp in glob.glob(os.path.join(META_DIR, "*.json")):
    stem = os.path.splitext(os.path.basename(fp))[0]
    try:
        meta_by_id[stem] = json.load(open(fp))
    except Exception:
        pass

GARMENT_SLOTS = ("dress", "top", "bottom", "outerwear")

def primary_garment(meta):
    # (colour, type) for the first non-empty garment slot, else (None, None)
    for slot in GARMENT_SLOTS:
        gtype = meta.get(slot)
        if gtype:
            return meta.get(f"{slot}_color"), gtype
    return None, None

def build_queries(meta):
    colour, gtype = primary_garment(meta)
    if not gtype:
        return None, None
    q_short = " ".join(w for w in [colour, gtype] if w).strip()
    env = meta.get("environment")
    q_medium = f"{q_short} in {env}" if env else q_short
    return q_short, q_medium

eval_set = []          # list of (id, q_short, q_medium)
skipped = 0
for rec in records:
    meta = meta_by_id.get(rec["id"])
    if meta is None:
        skipped += 1; continue
    qs, qm = build_queries(meta)
    if not qs:
        skipped += 1; continue
    eval_set.append((rec["id"], qs, qm))

print(f"eval queries: {len(eval_set)}   skipped (no garment/metadata): {skipped}")
print("\nexamples (query  |  indexed caption):")
id2rec = {rec["id"]: rec for rec in records}
for _id, qs, qm in eval_set[:3]:
    print(f"  q_short  : {qs!r}")
    print(f"  q_medium : {qm!r}")
    print(f"  caption  : {id2rec[_id]['caption']!r}\n")

eval queries: 461   skipped (no garment/metadata): 0

examples (query  |  indexed caption):
  q_short  : 'beige sleeveless dress'
  q_medium : 'beige sleeveless dress in forest path'
  caption  : 'A person wearing beige polka dot lace trim sleeveless dress, green sandals, with gold bracelet, posing, casual style, in forest path.'

  q_short  : 'gray long-sleeved dress'
  q_medium : 'gray long-sleeved dress in studio'
  caption  : 'A person wearing gray heathered long-sleeved dress, standing, minimalist casual style, in studio.'

  q_short  : 'black sleeveless dress'
  q_medium : 'black sleeveless dress in fashion runway'
  caption  : 'A person wearing black asymmetric sleeveless dress, black high heels, walking, elegant evening style, in fashion runway.'



## 2. Metrics

`rank_of_source` encodes the query **once**, then ranks it against both FAISS indexes with the
package's own `fused_candidates` + `rerank` (identical to `FusionRetriever.search`, just factored so
we can reuse one query vector across alpha values). The source's 1-indexed position in the top
`top_candidates` (=100) ranking is its rank; not in the top-100 -> a **miss** (`None`).

- **Recall@K** - fraction of queries whose source image is in the top-K.
- **MRR** - mean of `1/rank` (miss -> 0); reported over the top-100 window.
- **Median rank** - median position of the source among *found* queries.
- **%found** - fraction ranked within the top-100 at all.

In [3]:
def rank_of_source(q_vec, query_text, source_id, a):
    cands, _t, _i = fused_candidates(r.store.index, r.image_index, q_vec, records,
                                     alpha=a, top_candidates=cfg.top_candidates)
    ranked = rerank(query_text, cands, cfg)          # [(record, final, debug), ...] best-first
    for pos, (rec, _f, _d) in enumerate(ranked, 1):
        if rec["id"] == source_id:
            return pos
    return None

def recall_at_k(ranks, k):
    return sum(1 for x in ranks if x is not None and x <= k) / len(ranks)

def mrr(ranks):
    return sum((1.0 / x) if x is not None else 0.0 for x in ranks) / len(ranks)

def summarize(ranks, label):
    found = [x for x in ranks if x is not None]
    return {
        "eval": label,
        "N": len(ranks),
        "Recall@1": round(recall_at_k(ranks, 1), 4),
        "Recall@5": round(recall_at_k(ranks, 5), 4),
        "Recall@10": round(recall_at_k(ranks, 10), 4),
        "MRR": round(mrr(ranks), 4),
        "median_rank": (median(found) if found else None),
        "%found": round(len(found) / len(ranks), 4),
    }

## 3. Headline numbers (default config: alpha=0.5, rerank on)

We encode every query once up front (the only real cost - SigLIP text tower on CPU) and cache the
vectors so the alpha sweep in section 4 is just FAISS scans.

In [4]:
# cache query embeddings once (reused across alphas)
short_vecs  = {i: r.encoder.embed_query(qs) for i, (_id, qs, qm) in enumerate(eval_set)}
medium_vecs = {i: r.encoder.embed_query(qm) for i, (_id, qs, qm) in enumerate(eval_set)}
print(f"encoded {len(short_vecs)} short + {len(medium_vecs)} medium query vectors")

A_DEFAULT = cfg.fusion_alpha
ranks_short  = [rank_of_source(short_vecs[i],  qs, _id, A_DEFAULT) for i, (_id, qs, qm) in enumerate(eval_set)]
ranks_medium = [rank_of_source(medium_vecs[i], qm, _id, A_DEFAULT) for i, (_id, qs, qm) in enumerate(eval_set)]

headline = pd.DataFrame([
    summarize(ranks_short,  f"q_short  (a={A_DEFAULT})"),
    summarize(ranks_medium, f"q_medium (a={A_DEFAULT})"),
])
headline

encoded 461 short + 461 medium query vectors


,eval,N,Recall@1,Recall@5,Recall@10,MRR,median_rank,%found
0,q_short (a=0.5),461,0.4685,0.7701,0.8677,0.6002,2.0,0.9414
1,q_medium (a=0.5),461,0.7267,0.9393,0.9631,0.8154,1.0,0.9848


## 4. alpha sweep (image-only -> text-only)

`fused = a*text_sim + (1-a)*image_sim`. Sweeping `a` shows how much the caption-text tower vs. the
SigLIP image tower each contribute. Note the documented caveat: SigLIP text-text cosines are much
larger than text-image ones, so `a=0.5` is **not** a balanced 50/50 - the fused score is
text-dominated unless similarities are normalized per query. This table makes that concrete.

In [5]:
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]     # 0 = image-only, 1 = text-only
rows = []
for a in ALPHAS:
    ranks = [rank_of_source(medium_vecs[i], qm, _id, a) for i, (_id, qs, qm) in enumerate(eval_set)]
    s = summarize(ranks, f"q_medium")
    s["alpha"] = a
    rows.append(s)

sweep = pd.DataFrame(rows)[["alpha", "N", "Recall@1", "Recall@5", "Recall@10", "MRR", "median_rank", "%found"]]
sweep

,alpha,N,Recall@1,Recall@5,Recall@10,MRR,median_rank,%found
0,0.00,461,0.7158,0.9219,0.9566,0.8029,1.0,0.9761
1,0.25,461,0.7397,0.9436,0.9696,0.8253,1.0,0.9892
2,0.50,461,0.7267,0.9393,0.9631,0.8154,1.0,0.9848
3,0.75,461,0.6659,0.9024,0.9479,0.7668,1.0,0.9783
4,1.00,461,0.6117,0.8547,0.9262,0.7182,1.0,0.9653


## 5. Save results

Written to the repo root as `retrieval_metrics.csv` and `retrieval_metrics.md` for the write-up.
The `random baseline` row is the expected Recall@K if ranking were random over the corpus
(`K / N`), for context.

In [6]:
out = sweep.copy()
out.insert(0, "eval", "known-item q_medium")

# random baseline for context (expected recall of a uniform-random ranker)
baseline = {"eval": "random baseline", "alpha": None, "N": len(eval_set),
            "Recall@1": round(1/N, 4), "Recall@5": round(5/N, 4), "Recall@10": round(10/N, 4),
            "MRR": None, "median_rank": N//2, "%found": None}
final = pd.concat([headline.assign(alpha=A_DEFAULT),
                   out, pd.DataFrame([baseline])], ignore_index=True)
final = final[["eval", "alpha", "N", "Recall@1", "Recall@5", "Recall@10", "MRR", "median_rank", "%found"]]

csv_path = os.path.join(ROOT, "retrieval_metrics.csv")
md_path  = os.path.join(ROOT, "retrieval_metrics.md")
final.to_csv(csv_path, index=False)
with open(md_path, "w") as f:
    f.write("# Glance retrieval - known-item metrics\n\n")
    f.write("Known-item (self-)retrieval over {} indexed images. Relevance = the source image; "
            "queries are shortened attribute descriptions (see notebook section 1). "
            "No human labels involved.\n\n".format(N))
    f.write(final.to_markdown(index=False))
    f.write("\n")
print("wrote", csv_path)
print("wrote", md_path)
final

wrote /home/jitdarkfighter/Projects/Glance/retrieval_metrics.csv
wrote /home/jitdarkfighter/Projects/Glance/retrieval_metrics.md


,eval,alpha,N,Recall@1,Recall@5,Recall@10,MRR,median_rank,%found
0,q_short (a=0.5),0.5,461,0.4685,0.7701,0.8677,0.6002,2.0,0.9414
1,q_medium (a=0.5),0.5,461,0.7267,0.9393,0.9631,0.8154,1.0,0.9848
2,known-item q_medium,0.0,461,0.7158,0.9219,0.9566,0.8029,1.0,0.9761
3,known-item q_medium,0.25,461,0.7397,0.9436,0.9696,0.8253,1.0,0.9892
4,known-item q_medium,0.5,461,0.7267,0.9393,0.9631,0.8154,1.0,0.9848
5,known-item q_medium,0.75,461,0.6659,0.9024,0.9479,0.7668,1.0,0.9783
6,known-item q_medium,1.0,461,0.6117,0.8547,0.9262,0.7182,1.0,0.9653
7,random baseline,None,461,0.0022,0.0108,0.0217,None,230.0,None
